# 04_statistical_analysis_improved


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, pearsonr, f_oneway
import statsmodels.api as sm
from IPython.display import display, Markdown


In [ ]:
# Load dataset
file_path = "/content/ai_job_trends_dataset 2.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

print("\nColumns:")
for col in df.columns:
    print("-", col)


## Step 1: Use the exact columns from the CSV

In [ ]:
salary_col = "Median Salary (USD)"
risk_col = "Automation Risk (%)"
ai_col = "AI Impact Level"
education_col = "Required Education"
openings_2024_col = "Job Openings (2024)"
openings_2030_col = "Projected Openings (2030)"
experience_col = "Experience Required (Years)"
remote_col = "Remote Work Ratio (%)"
diversity_col = "Gender Diversity (%)"


## Step 2: Create analysis variables

We create:

- **Growth (%)** = change in openings from 2024 to 2030
- **Decline** = opening loss when projected openings are lower


In [ ]:
# Convert numeric columns
numeric_cols = [
    salary_col, risk_col, openings_2024_col, openings_2030_col,
    experience_col, remote_col, diversity_col
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Create growth and decline
df["Growth (%)"] = ((df[openings_2030_col] - df[openings_2024_col]) / df[openings_2024_col]) * 100
df["Decline"] = (df[openings_2024_col] - df[openings_2030_col]).clip(lower=0)

print(df[[salary_col, risk_col, openings_2024_col, openings_2030_col, "Growth (%)", "Decline"]].head())


## Step 3: Correlation analysis

### Objective
Study the relationship between:
- salary
- growth
- risk

### Dashboard linkage
These results can support a dashboard section called:
**"Relationship Overview: Salary, Growth, and Risk"**


In [ ]:
correlation_data = df[[salary_col, "Growth (%)", risk_col]].dropna()

print("Correlation Matrix")
display(correlation_data.corr())

print("\nPearson correlation tests")

r1, p1 = pearsonr(correlation_data[salary_col], correlation_data["Growth (%)"])
print(f"Salary vs Growth: r = {r1:.4f}, p = {p1:.4f}")

r2, p2 = pearsonr(correlation_data[salary_col], correlation_data[risk_col])
print(f"Salary vs Risk: r = {r2:.4f}, p = {p2:.4f}")

r3, p3 = pearsonr(correlation_data["Growth (%)"], correlation_data[risk_col])
print(f"Growth vs Risk: r = {r3:.4f}, p = {p3:.4f}")


In [ ]:
print("Business interpretation:")

if p1 < 0.05:
    print("- Salary vs Growth is statistically significant.")
else:
    print("- Salary vs Growth is not statistically significant.")

if p2 < 0.05:
    print("- Salary vs Risk is statistically significant.")
else:
    print("- Salary vs Risk is not statistically significant.")

if p3 < 0.05:
    print("- Growth vs Risk is statistically significant.")
else:
    print("- Growth vs Risk is not statistically significant.")

print("\nUse this in dashboard:")
print("- Show correlation heatmap")
print("- Show 3 KPI notes: Salary-Growth, Salary-Risk, Growth-Risk")


## Step 4: Prepare AI groups

For simple comparison tests:

- **High AI** = 1
- **Low AI** = 0

Rows with **Medium AI** are excluded only from AI group t-tests.


In [ ]:
df[ai_col] = df[ai_col].astype(str).str.strip().str.lower()

df["AI_Group"] = np.nan
df.loc[df[ai_col] == "high", "AI_Group"] = 1
df.loc[df[ai_col] == "low", "AI_Group"] = 0

print("AI Impact Level counts:")
print(df[ai_col].value_counts(dropna=False))

print("\nAI_Group counts used in t-tests:")
print(df["AI_Group"].value_counts(dropna=False))


## Step 5: Hypothesis Test 1 — AI vs Growth

### Hypotheses
- **H0:** Mean growth is the same for low-AI and high-AI jobs
- **H1:** Mean growth is different for low-AI and high-AI jobs

### Dashboard linkage
This can support a dashboard card:
**"Does AI impact job growth?"**


In [ ]:
ai_growth_data = df[["AI_Group", "Growth (%)"]].dropna()

growth_high_ai = ai_growth_data[ai_growth_data["AI_Group"] == 1]["Growth (%)"]
growth_low_ai = ai_growth_data[ai_growth_data["AI_Group"] == 0]["Growth (%)"]

t_stat, p_val = ttest_ind(growth_high_ai, growth_low_ai, equal_var=False)

print("AI vs Growth")
print("High AI mean growth:", growth_high_ai.mean())
print("Low AI mean growth:", growth_low_ai.mean())
print(f"t-statistic = {t_stat:.4f}")
print(f"p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Interpretation: Reject H0. AI level has a statistically significant relationship with growth.")
else:
    print("Interpretation: Fail to reject H0. AI level does not show a statistically significant difference in growth.")

print("\nBusiness insight:")
if growth_high_ai.mean() > growth_low_ai.mean():
    print("- High-AI jobs are growing faster on average.")
else:
    print("- Low-AI jobs are growing faster on average.")


## Step 6: Hypothesis Test 2 — AI vs Salary

### Hypotheses
- **H0:** Mean salary is the same for low-AI and high-AI jobs
- **H1:** Mean salary is different for low-AI and high-AI jobs

### Dashboard linkage
This can support a dashboard card:
**"Does AI impact salary?"**


In [ ]:
ai_salary_data = df[["AI_Group", salary_col]].dropna()

salary_high_ai = ai_salary_data[ai_salary_data["AI_Group"] == 1][salary_col]
salary_low_ai = ai_salary_data[ai_salary_data["AI_Group"] == 0][salary_col]

t_stat, p_val = ttest_ind(salary_high_ai, salary_low_ai, equal_var=False)

print("AI vs Salary")
print("High AI mean salary:", salary_high_ai.mean())
print("Low AI mean salary:", salary_low_ai.mean())
print(f"t-statistic = {t_stat:.4f}")
print(f"p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Interpretation: Reject H0. AI level has a statistically significant relationship with salary.")
else:
    print("Interpretation: Fail to reject H0. AI level does not show a statistically significant difference in salary.")

print("\nBusiness insight:")
if salary_high_ai.mean() > salary_low_ai.mean():
    print("- High-AI jobs pay more on average.")
else:
    print("- Low-AI jobs pay more on average.")


## Step 7: Hypothesis Test 3 — Risk vs Decline

### Hypotheses
- **H0:** There is no linear relationship between automation risk and decline
- **H1:** There is a linear relationship between automation risk and decline

### Dashboard linkage
This can support a dashboard card:
**"Does automation risk connect with job decline?"**


In [ ]:
risk_decline_data = df[[risk_col, "Decline"]].dropna()

r, p = pearsonr(risk_decline_data[risk_col], risk_decline_data["Decline"])

print("Risk vs Decline")
print(f"correlation = {r:.4f}")
print(f"p-value = {p:.4f}")

if p < 0.05:
    print("Interpretation: Reject H0. Risk is significantly related to decline.")
else:
    print("Interpretation: Fail to reject H0. Risk is not significantly related to decline.")

print("\nBusiness insight:")
if r > 0:
    print("- Higher automation risk is associated with larger decline.")
else:
    print("- Higher automation risk is associated with lower decline.")


## Step 8: Hypothesis Test 4 — Education vs Salary

### Hypotheses
- **H0:** Mean salary is the same across education levels
- **H1:** At least one education level has a different mean salary

### Dashboard linkage
This can support a dashboard card:
**"How education level changes salary"**


In [ ]:
education_salary_data = df[[education_col, salary_col]].dropna()

groups = []
group_names = []

for name, group in education_salary_data.groupby(education_col):
    if len(group) > 1:
        groups.append(group[salary_col].values)
        group_names.append(name)

f_stat, p_val = f_oneway(*groups)

print("Education vs Salary")
print("Education groups:", group_names)
print(f"F-statistic = {f_stat:.4f}")
print(f"p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Interpretation: Reject H0. Salary differs across education levels.")
else:
    print("Interpretation: Fail to reject H0. Salary does not significantly differ across education levels.")

print("\nBusiness insight:")
print("- This test helps explain whether education requirements are linked to salary differences.")


## Step 9: Create numeric variables for regression

We convert text categories into simple numeric scores.


In [ ]:
# AI score for regression
ai_map = {"low": 1, "medium": 2, "high": 3}
df["AI_Impact_Score"] = df[ai_col].map(ai_map)

# Education score for regression
education_map = {
    "High School": 1,
    "Associate": 2,
    "Bachelor's": 3,
    "Master's": 4,
    "PhD": 5
}
df["Education_Score"] = df[education_col].map(education_map)

print(df[[ai_col, "AI_Impact_Score", education_col, "Education_Score"]].head())


## Step 10: Regression Model 1 — Salary Drivers

### Objective
Predict salary using:
- AI impact
- automation risk
- experience
- remote work ratio
- gender diversity
- education score
- growth

### Dashboard linkage
This can support a dashboard section:
**"Main salary drivers"**


In [ ]:
salary_regression_data = df[
    [
        salary_col,
        "AI_Impact_Score",
        risk_col,
        experience_col,
        remote_col,
        diversity_col,
        "Education_Score",
        "Growth (%)"
    ]
].dropna()

X = salary_regression_data[
    [
        "AI_Impact_Score",
        risk_col,
        experience_col,
        remote_col,
        diversity_col,
        "Education_Score",
        "Growth (%)"
    ]
]
y = salary_regression_data[salary_col]

X = sm.add_constant(X)
salary_model = sm.OLS(y, X).fit()

print("Salary Regression Results")
print(salary_model.summary())


In [ ]:
print("Business interpretation for salary regression:")
print("- Check coefficient signs to see whether each variable increases or decreases salary.")
print("- Check p-values to see which salary drivers are statistically significant.")
print("- Use R-squared to explain how much salary variation is captured by the model.")
print("- Dashboard idea: show top positive driver and top negative driver.")


## Step 11: Regression Model 2 — Growth Drivers

### Objective
Predict growth using:
- salary
- AI impact
- automation risk
- experience
- remote work ratio
- gender diversity
- education score

### Dashboard linkage
This can support a dashboard section:
**"Main growth drivers"**


In [ ]:
growth_regression_data = df[
    [
        "Growth (%)",
        salary_col,
        "AI_Impact_Score",
        risk_col,
        experience_col,
        remote_col,
        diversity_col,
        "Education_Score"
    ]
].dropna()

X = growth_regression_data[
    [
        salary_col,
        "AI_Impact_Score",
        risk_col,
        experience_col,
        remote_col,
        diversity_col,
        "Education_Score"
    ]
]
y = growth_regression_data["Growth (%)"]

X = sm.add_constant(X)
growth_model = sm.OLS(y, X).fit()

print("Growth Regression Results")
print(growth_model.summary())


In [ ]:
print("Business interpretation for growth regression:")
print("- Check coefficient signs to identify what increases or reduces job growth.")
print("- Check p-values to identify statistically meaningful growth drivers.")
print("- Dashboard idea: show strongest growth driver and strongest drag on growth.")


## Step 12: Final reporting guide

Use this structure in your final write-up:

1. State the hypothesis
2. Show test statistic and p-value
3. Say whether the result is significant
4. Explain the business meaning
5. Mention where it fits in the dashboard

This makes the notebook more complete, more rigorous, and better for scoring.
